# Hazard Analysis and Risk Assessment (HARA)

**Module 03 · Functional Safety (ISO 26262)** · Notebook 2 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/03_Functional_Safety_ISO26262/02_hara.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~60 min | [01 ISO 26262 fundamentals](01_iso26262_fundamentals.ipynb) | ISO 26262-3:2018, ISO 26262-9:2018 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Carry out the HARA steps of ISO 26262-3 (clause 6) for an automated emergency braking (AEB) function
2. Write an item definition at vehicle level and build an operational-situation catalogue with exposure rationale
3. Formulate hazards as vehicle-level malfunctioning behaviours, independent of their cause
4. Classify S, E and C with the wording of the standard and a written rationale, and determine the ASIL
5. Derive vehicle-level safety goals with safe states and FTTI, and check a mechanism's FDTI + FRTI against the FTTI
6. Apply ASIL decomposition (D → B(D) + B(D)) and state the independence it requires

## 1. Introduction to HARA

**Hazard analysis and risk assessment (HARA)** is the method in ISO 26262-3:2018, clause 6, that identifies and classifies the **hazardous events** caused by *malfunctioning behaviour* of an item, and formulates **safety goals** with an ASIL to prevent or mitigate them.

### Purpose of HARA

- Identify hazardous events that could result from malfunctioning behaviour of the item
- Classify each hazardous event by Severity, Exposure and Controllability
- Determine the Automotive Safety Integrity Level (ASIL)
- Formulate safety goals (top-level safety requirements) and their attributes

### HARA in the safety lifecycle

HARA is performed in the **concept phase** (ISO 26262-3):

```
Item definition (cl. 5) --> HARA (cl. 6) --> Functional safety concept (cl. 7) --> Technical safety concept (ISO 26262-4)
```

The outputs of HARA drive all subsequent safety activities.

> **Three rules for every HARA**
> 1. It analyses the **item at vehicle level**. Hazards are vehicle behaviours ("unintended deceleration"), not sensor or algorithm outputs ("false positive").
> 2. It is carried out **without considering the safety mechanisms** of the item under analysis. Their effect is credited later, in the safety concept.
> 3. Each hazardous event is rated for the **worst credible** operational situation, and every S, E and C rating has a **documented rationale** (see SAE J2980 for guidance).

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

rng = np.random.default_rng(42)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 100})
print("Setup complete.")

## 2. HARA process overview

1. **Item definition**: define the function at vehicle level, its boundary and interfaces
2. **Situation analysis**: identify operational situations and operating modes
3. **Hazard identification**: identify vehicle-level malfunctioning behaviours and the resulting hazardous events
4. **Hazard classification**: classify S, E and C for each hazardous event, with rationale
5. **ASIL determination**: determine the ASIL from S, E and C
6. **Safety goal definition**: formulate safety goals for hazardous events rated ASIL A–D

In [ ]:
def visualise_hara_process():
    """Visualise the HARA process flow."""
    fig, ax = plt.subplots(figsize=(14, 3.6))
    okabe_ito = ["#0072B2", "#009E73", "#D55E00", "#CC79A7", "#E69F00", "#56B4E9"]
    steps = ["Item\ndefinition", "Situation\nanalysis", "Hazard\nidentification",
             "Classification\n(S, E, C)", "ASIL\ndetermination", "Safety goal\ndefinition"]
    xs = np.linspace(0.5, 5.5, len(steps))
    for i, (label, colour) in enumerate(zip(steps, okabe_ito)):
        ax.add_patch(FancyBboxPatch((xs[i] - 0.36, 0.3), 0.72, 0.4, boxstyle="round,pad=0.05",
                                    facecolor=colour, edgecolor="black", linewidth=1.5))
        ax.text(xs[i], 0.5, label, ha="center", va="center", fontsize=10, fontweight="bold", color="black")
        if i < len(steps) - 1:
            ax.annotate("", xy=(xs[i + 1] - 0.42, 0.5), xytext=(xs[i] + 0.42, 0.5),
                        arrowprops=dict(arrowstyle="->", lw=2))
    ax.set_xlim(0, 6); ax.set_ylim(0.1, 0.9); ax.axis("off")
    ax.set_title("HARA process flow (ISO 26262-3:2018, clause 6)", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
    return steps

steps = visualise_hara_process()
print("--- HARA process flow (text summary) ---")
for i, s in enumerate(steps, 1):
    print(f"  Step {i}: {s.replace(chr(10), ' ')}")
print("  Each step feeds into the next in sequence.")

## 3. Step 1: Item definition

Before the HARA, the **item** is defined (ISO 26262-3:2018, clause 5). An item is a system or combination of systems that implements a function **at vehicle level**. A perception subsystem on its own has no actuator, so it cannot cause a vehicle-level hazardous event. The item here is therefore the whole AEB function, including the brake request.

### Item definition template

| Attribute | Description |
|---|---|
| **Item name** | Name of the function at vehicle level |
| **Functional description** | What the item does, as seen from the vehicle |
| **Boundary** | Which elements are included or excluded |
| **Interfaces** | Inputs, outputs and interactions with other items |
| **Operating modes** | E.g. active, standby, degraded |
| **Operating conditions** | Speed range, environment, legal requirements |
| **Assumptions** | E.g. driver role, behaviour of neighbouring items |

### Example: AEB function of a passenger car (SAE Level 1 driver assistance)

In [ ]:
item_definition = {
    "Item name": "Automated emergency braking (AEB) function",
    "Functional description": ("Detects vehicles, pedestrians and cyclists in the vehicle path and, if a collision is "
                               "imminent and the driver does not react, requests an automatic deceleration of up to "
                               "full braking (about 10 m/s^2)."),
    "Boundary": [
        "Includes: front camera and radar, perception and object fusion, threat assessment and decision logic",
        "Includes: brake request to the brake system (interface signal) and driver warning request",
        "Excludes: brake system / ESC hydraulics (separate item; provides the requested deceleration)",
        "Excludes: instrument-cluster HMI hardware (separate item; displays warnings)",
    ],
    "Interfaces": {
        "Inputs": ["Camera images", "Radar targets", "Vehicle speed, yaw rate", "Driver brake-pedal and steering activity"],
        "Outputs": ["Deceleration request to brake system", "Driver warning request", "AEB status"],
    },
    "Operating modes": ["Active (10-130 km/h)", "Standby (outside speed range or switched off)", "Unavailable (fault detected)"],
    "Operating conditions": ["Public roads: urban, rural, motorway", "Day and night, dry and wet roads"],
    "Assumptions": ["SAE Level 1: the driver remains responsible for monitoring and braking",
                    "The brake system executes a valid deceleration request"],
}

print(f"Item definition: {item_definition['Item name']}")
print("=" * 60)
for key, value in item_definition.items():
    if key == "Item name":
        continue
    print(f"\n{key}:")
    if isinstance(value, list):
        for v in value:
            print(f"  - {v}")
    elif isinstance(value, dict):
        for k, v in value.items():
            print(f"  {k}: {', '.join(v)}")
    else:
        print(f"  {value}")

## 4. Step 2: Situation analysis

Identify the **operational situations** in which a malfunction of the item could lead to harm. Consider road type, speed, traffic participants (including those behind the vehicle), environmental conditions and the driver's state.

The **exposure** class belongs to the operational situation and is rated either by **duration** (share of average operating time) or by **frequency** (ISO 26262-3:2018, Annex B):

| Class | Description | Duration | Frequency |
|---|---|---|---|
| E1 | Very low probability | Not specified | Less often than once a year for the great majority of drivers |
| E2 | Low probability | < 1 % of average operating time | A few times a year for the great majority of drivers |
| E3 | Medium probability | 1 % to 10 % | Once a month or more often for an average driver |
| E4 | High probability | > 10 % | All or almost every drive on average |

E0 ("incredible") situations receive no ASIL. Choose the basis (duration or frequency) that fits the hazard: an *unintended* activation can strike at any time while driving, so a **duration** basis fits; an *omission* only matters when the function is needed, so a **frequency** basis fits.

> **Illustrative data:** the exposure ratings below are plausible teaching values. A real HARA backs them with field or naturalistic-driving data.

In [ ]:
situations = pd.DataFrame([
    {"ID": "OS-01", "Situation": "Urban road, 30-50 km/h, pedestrian steps into the vehicle path, collision imminent",
     "E": "E2", "Basis": "frequency",
     "Rationale": "Emergency-braking conflicts with pedestrians occur a few times a year at most for most drivers"},
    {"ID": "OS-02", "Situation": "Urban road, 30-50 km/h, following vehicle close behind",
     "E": "E4", "Basis": "duration",
     "Rationale": "Urban driving with a follower is well above 10 % of average operating time"},
    {"ID": "OS-03", "Situation": "Motorway, 100-130 km/h, following vehicle at short headway (about 1 s)",
     "E": "E4", "Basis": "duration",
     "Rationale": "Assumed > 10 % of operating time for the target market; confirm with field data"},
    {"ID": "OS-04", "Situation": "Motorway or rural road, 80-130 km/h, approaching the stationary end of a traffic jam",
     "E": "E3", "Basis": "frequency",
     "Rationale": "Approaching queues at speed happens about once a month or more for an average driver"},
])
display(situations)

print("--- Operational situations (text summary) ---")
for _, r in situations.iterrows():
    print(f"  {r.ID}: {r.Situation} -> {r.E} (by {r.Basis}); {r.Rationale}")

## 5. Step 3: Hazard identification

Hazards are identified as **vehicle-level malfunctioning behaviours** of the item. Guide words help: *no / unintended / too early / too late / too much / too little / too long*. For AEB:

| ID | Malfunctioning behaviour (vehicle level) | Guide word |
|---|---|---|
| MB-01 | No or insufficient AEB deceleration when a collision is imminent | no / too little / too late (omission) |
| MB-02 | Unintended AEB deceleration (up to full braking) without a collision threat | unintended (commission) |
| MB-03 | AEB deceleration stronger or longer than required (not released after the threat has passed) | too much / too long |

A **hazardous event** is a malfunctioning behaviour combined with an operational situation. The HARA does **not** ask *why* the malfunction occurs. Perception errors are possible **causes**, and they belong to different standards depending on their origin:

| Perception error (cause) | Leads to | If caused by an E/E fault → ISO 26262 | If caused by a performance limitation → ISO 21448 / ISO/PAS 8800 |
|---|---|---|---|
| False negative (object present, not detected) | MB-01 | Camera ECU RAM bit flip corrupts the object list; frozen image (stuck frame) | Low sun glare, occlusion, pedestrian type under-represented in training data |
| Late detection | MB-01 | Task overrun because of a hardware fault | Long confirmation time needed in rain |
| False positive (ghost object) | MB-02 | Corrupted radar target message | Shadow, manhole cover or reflection classified as an obstacle |
| Wrong position or distance | MB-01, MB-02, MB-03 | Calibration data corrupted in memory | Monocular depth error at long range |
| Tracking loss | MB-01 | Message loss on the vehicle bus | Prolonged occlusion |

In [ ]:
malfunctions = pd.DataFrame([
    {"MB": "MB-01", "Malfunctioning behaviour": "No or insufficient AEB deceleration when a collision is imminent",
     "Hazard": "Collision with a pedestrian or vehicle ahead that AEB should have mitigated",
     "Potential harm": "Injury to pedestrian or occupants"},
    {"MB": "MB-02", "Malfunctioning behaviour": "Unintended AEB deceleration without a collision threat",
     "Hazard": "Rear-end collision by following traffic; loss of vehicle stability",
     "Potential harm": "Injury to occupants of both vehicles"},
    {"MB": "MB-03", "Malfunctioning behaviour": "AEB deceleration stronger or longer than required",
     "Hazard": "Rear-end collision after the threat has passed",
     "Potential harm": "Injury to occupants of both vehicles"},
])
display(malfunctions)

print("--- Malfunctioning behaviours (text summary) ---")
for _, r in malfunctions.iterrows():
    print(f"  {r.MB}: {r['Malfunctioning behaviour']} -> hazard: {r.Hazard}; harm: {r['Potential harm']}")

## 6. Step 4: Hazard classification (S, E, C)

Classes as in ISO 26262-3:2018, Annex B.

### Severity (S): extent of harm to each person potentially affected

| Class | Description | Informative AIS guidance | Example |
|---|---|---|---|
| **S0** | No injuries | AIS 0, or < 10 % probability of AIS 1–6 | Damage to property only |
| **S1** | Light and moderate injuries | > 10 % probability of AIS 1–6 (not S2/S3) | Low-speed rear-end impact (whiplash) |
| **S2** | Severe and life-threatening injuries (survival probable) | > 10 % probability of AIS 3–6 (not S3) | Side impact at medium speed |
| **S3** | Life-threatening injuries (survival uncertain), fatal injuries | > 10 % probability of AIS 5–6 | Pedestrian struck at 50 km/h |

### Exposure (E): see the table in section 4 (duration *or* frequency; E0 = incredible, no percentage)

### Controllability (C): ability of the driver **or other persons at risk** to avoid the specific harm

| Class | Description | Guidance |
|---|---|---|
| **C0** | Controllable in general | No percentage; no ASIL assigned |
| **C1** | Simply controllable | 99 % or more of drivers or other traffic participants are usually able to avoid harm |
| **C2** | Normally controllable | 90 % or more are usually able to avoid harm |
| **C3** | Difficult to control or uncontrollable | Less than 90 % are usually able, or barely able, to avoid harm |

For unintended braking, the persons who must "control" the situation are mainly the **drivers of following vehicles**.

In [ ]:
def visualise_sec_scales():
    """Visualise the S, E, C classes as ordered scales."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    scales = {
        "Severity (S)": ["S0 No injuries", "S1 Light/moderate", "S2 Severe (survival probable)", "S3 Life-threatening/fatal"],
        "Exposure (E)": ["E0 Incredible", "E1 Very low", "E2 Low (<1 % / few per year)",
                         "E3 Medium (1-10 % / monthly)", "E4 High (>10 % / almost every drive)"],
        "Controllability (C)": ["C0 Controllable in general", "C1 Simply (>=99 %)", "C2 Normally (>=90 %)",
                                "C3 Difficult/uncontrollable (<90 %)"],
    }
    for ax, (title, labels) in zip(axes, scales.items()):
        n = len(labels)
        ax.barh(range(n), range(1, n + 1), color=plt.cm.viridis(np.linspace(0.15, 0.9, n)), edgecolor="black")
        ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel("Class index (higher = contributes to a higher ASIL)")
        ax.set_title(title, fontweight="bold"); ax.invert_yaxis()
    plt.suptitle("ISO 26262-3:2018 classification parameters", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
    return scales

scales = visualise_sec_scales()
print("--- S, E, C classes (text summary) ---")
for title, labels in scales.items():
    print(f"  {title}: " + "; ".join(labels))

## 7. Step 5: ASIL determination

The ASIL is read from ISO 26262-3:2018 Table 4. The function below reproduces the table exactly, as `avsafety.iso26262.determine_asil` does. Invalid inputs raise an error instead of silently returning QM.

- **QM**: no ASIL; quality management is sufficient
- **ASIL A**: least stringent … **ASIL D**: most stringent

In [ ]:
# ISO 26262-3:2018 Table 4 -- ASIL determination. ASIL_TABLE[S][E] -> (C1, C2, C3)
# Identical to avsafety.iso26262.ASIL_TABLE (see the repository's avsafety/ package).
ASIL_ORDER = ("QM", "A", "B", "C", "D")
ASIL_TABLE = {
    "S1": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "QM"), "E3": ("QM", "QM", "A"), "E4": ("QM", "A", "B")},
    "S2": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "A"), "E3": ("QM", "A", "B"), "E4": ("A", "B", "C")},
    "S3": {"E1": ("QM", "QM", "A"), "E2": ("QM", "A", "B"), "E3": ("A", "B", "C"), "E4": ("B", "C", "D")},
}
_VALID = {"S": ("S0", "S1", "S2", "S3"), "E": ("E0", "E1", "E2", "E3", "E4"), "C": ("C0", "C1", "C2", "C3")}


def determine_asil(severity: str, exposure: str, controllability: str) -> str:
    """Return the ASIL ("QM", "A".."D") of a hazardous event (ISO 26262-3:2018 Table 4).

    S0, E0 or C0 mean that no ASIL is assigned (returned as "QM").
    Invalid classes raise ValueError -- a typo must never silently become "QM".
    The same function is available as avsafety.iso26262.determine_asil.
    """
    s, e, c = (str(x).strip().upper() for x in (severity, exposure, controllability))
    for val, key in ((s, "S"), (e, "E"), (c, "C")):
        if val not in _VALID[key]:
            raise ValueError(f"invalid {key} class: {val!r}; expected one of {_VALID[key]}")
    if s == "S0" or e == "E0" or c == "C0":
        return "QM"
    return ASIL_TABLE[s][e][int(c[1]) - 1]


def asil_rank(asil: str) -> int:
    """Numeric rank of an ASIL (QM=0 ... D=4)."""
    return ASIL_ORDER.index(asil)


def visualise_asil_matrix():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, sev in zip(axes, ["S1", "S2", "S3"]):
        m = np.array([[asil_rank(a) for a in ASIL_TABLE[sev][e]] for e in ["E1", "E2", "E3", "E4"]])
        ax.imshow(m, cmap="viridis", vmin=0, vmax=4, aspect="auto")
        for i, e in enumerate(["E1", "E2", "E3", "E4"]):
            for j, a in enumerate(ASIL_TABLE[sev][e]):
                ax.text(j, i, a, ha="center", va="center", fontsize=14, fontweight="bold",
                        color="white" if asil_rank(a) < 3 else "black")
        ax.set_xticks(range(3)); ax.set_xticklabels(["C1", "C2", "C3"])
        ax.set_yticks(range(4)); ax.set_yticklabels(["E1", "E2", "E3", "E4"])
        ax.set_xlabel("Controllability class"); ax.set_ylabel("Exposure class")
        ax.set_title(f"Severity = {sev}", fontsize=12, fontweight="bold")
    plt.suptitle("ASIL determination, ISO 26262-3:2018 Table 4", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

visualise_asil_matrix()

print("--- ASIL determination table (text summary) ---")
for sev in ["S1", "S2", "S3"]:
    for exp in ["E1", "E2", "E3", "E4"]:
        c1, c2, c3 = ASIL_TABLE[sev][exp]
        print(f"  {sev} {exp}: C1={c1}, C2={c2}, C3={c3}")

try:
    determine_asil("S3", "E4", "C4")
except ValueError as err:
    print(f"\nInvalid input is rejected: {err}")

## 8. Complete HARA example

Each hazardous event (HE) combines a malfunctioning behaviour with an operational situation, and every rating carries its rationale.

> **Illustrative data:** these are teaching ratings with plausible rationale, not the HARA of a real product. Published AEB analyses differ, especially in the controllability of omission for Level 1 systems: some argue that an omission is QM, because the vehicle then behaves like one without AEB.

In [ ]:
hara_complete = pd.DataFrame([
    {"HE": "HE-01", "MB": "MB-01", "OS": "OS-01", "S": "S3", "E": "E2", "C": "C2",
     "S rationale": "Pedestrian struck at up to 50 km/h: survival uncertain",
     "C rationale": "Level 1: the driver remains responsible for braking; most attentive drivers brake themselves"},
    {"HE": "HE-02", "MB": "MB-01", "OS": "OS-04", "S": "S3", "E": "E3", "C": "C2",
     "S rationale": "Impact into a stationary vehicle at high speed",
     "C rationale": "Driver monitors the road and normally brakes for a traffic jam"},
    {"HE": "HE-03", "MB": "MB-02", "OS": "OS-03", "S": "S3", "E": "E4", "C": "C3",
     "S rationale": "Following vehicle, possibly a heavy goods vehicle, hits at a high speed difference",
     "C rationale": "Sudden full braking without warning at about 1 s headway: fewer than 90 % of following drivers can avoid the impact"},
    {"HE": "HE-04", "MB": "MB-02", "OS": "OS-02", "S": "S1", "E": "E4", "C": "C2",
     "S rationale": "Low-speed rear-end impact: light to moderate injuries",
     "C rationale": "Following drivers at urban speeds can normally brake in time"},
])
hara_complete = hara_complete.merge(situations[["ID", "Situation"]], left_on="OS", right_on="ID").drop(columns="ID")
hara_complete["ASIL"] = [determine_asil(r.S, r.E, r.C) for r in hara_complete.itertuples()]

display(hara_complete[["HE", "MB", "OS", "S", "E", "C", "ASIL", "S rationale", "C rationale"]])

print("--- Complete HARA (text summary) ---")
for _, r in hara_complete.iterrows():
    print(f"  {r.HE}: {r.MB} in {r.OS} ({r.Situation}) | {r.S}/{r.E}/{r.C} -> ASIL {r.ASIL}")
    print(f"      S: {r['S rationale']}; C: {r['C rationale']}")

# Sensitivity: how much does one class change the result?
print("\nSensitivity of HE-03 to the controllability rating:")
for c in ["C1", "C2", "C3"]:
    print(f"  S3/E4/{c} -> ASIL {determine_asil('S3', 'E4', c)}")

In [ ]:
def visualise_asil_distribution(df):
    """Bar chart of ASIL per hazardous event and count per ASIL."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ranks = df["ASIL"].map(asil_rank)
    bars = ax1.bar(df["HE"], ranks, color=plt.cm.viridis(ranks / 4), edgecolor="black")
    for b, a in zip(bars, df["ASIL"]):
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.08, a, ha="center", fontweight="bold")
    ax1.set_yticks(range(5)); ax1.set_yticklabels(ASIL_ORDER); ax1.set_ylim(0, 4.6)
    ax1.set_xlabel("Hazardous event"); ax1.set_ylabel("ASIL"); ax1.set_title("ASIL per hazardous event")
    counts = df["ASIL"].value_counts().reindex(ASIL_ORDER, fill_value=0)
    ax2.bar(counts.index, counts.values, color=plt.cm.viridis(np.arange(5) / 4), edgecolor="black")
    ax2.set_xlabel("ASIL"); ax2.set_ylabel("Number of hazardous events"); ax2.set_title("ASIL distribution")
    plt.tight_layout(); plt.show()
    return counts

counts = visualise_asil_distribution(hara_complete)
print("--- ASIL distribution (text summary) ---")
for level, n in counts.items():
    if n:
        print(f"  {level}: {n} hazardous event(s) ({100 * n / len(hara_complete):.0f} %)")
print(f"  Total hazardous events analysed: {len(hara_complete)}")

## 9. Step 6: Safety goal definition

A **safety goal** is a top-level safety requirement resulting from the HARA, formulated at **vehicle level**. Similar hazardous events may be combined into one goal, which then receives the **highest ASIL** among them.

### Safety goal attributes and timing terms (ISO 26262-1:2018)

| Attribute | Meaning |
|---|---|
| **ID, statement** | Vehicle-level goal, e.g. "Avoid unintended AEB deceleration" |
| **ASIL** | Highest ASIL of the hazardous events it covers |
| **Safe state** | Operating mode without an unreasonable level of risk, e.g. "AEB function switched off, driver warned" (an intervention such as "emergency braking activated" is *not* a safe state) |
| **FTTI**, fault tolerant time interval | Minimum time span from the occurrence of a fault in the item to a possible occurrence of a hazardous event, **if the safety mechanisms are not activated** |
| **FDTI**, fault detection time interval | Time span from the occurrence of a fault to its detection |
| **FRTI**, fault reaction time interval | Time span from the detection of a fault to reaching a safe state (or emergency operation) |

A safety mechanism is fast enough if its **fault handling time interval FHTI = FDTI + FRTI ≤ FTTI**. Commission hazards have an FTTI too: "N/A" is not an acceptable entry.

> **Illustrative data:** the FTTI and the mechanism timings below are placeholders. Real values are derived from vehicle dynamics (deceleration ramp, tolerable duration of unintended deceleration) and confirmed by vehicle tests.

In [ ]:
SG_CATALOGUE = {
    "MB-01": ("SG-02", "Avoid omission of AEB deceleration when a collision with a pedestrian or a stationary vehicle is imminent",
              "AEB unavailable, driver warned; driver remains responsible for braking", 500),
    "MB-02": ("SG-01", "Avoid unintended AEB deceleration (illustrative limit: > 3 m/s^2 for > 300 ms)",
              "AEB brake request inhibited, AEB-unavailable warning; driver keeps full control", 300),
}


def derive_safety_goals(hara_df):
    """Group hazardous events by malfunctioning behaviour; ASIL = highest ASIL of the group."""
    rows = []
    for mb, grp in hara_df[hara_df["ASIL"] != "QM"].groupby("MB"):
        sg_id, text, safe_state, ftti = SG_CATALOGUE.get(
            mb, (f"SG-{mb}", "TBD: formulate a vehicle-level goal", "TBD", None))
        rows.append({"SG": sg_id, "Safety goal": text, "ASIL": max(grp["ASIL"], key=asil_rank),
                     "Covers": ", ".join(grp["HE"]), "Safe state": safe_state, "FTTI (ms)": ftti})
    return pd.DataFrame(rows).sort_values("SG").reset_index(drop=True)


safety_goals_df = derive_safety_goals(hara_complete)
display(safety_goals_df)

print("--- Safety goals (text summary) ---")
for _, r in safety_goals_df.iterrows():
    print(f"  {r.SG} [ASIL {r.ASIL}] {r['Safety goal']} (covers {r.Covers})")
    print(f"      Safe state: {r['Safe state']} | FTTI: {r['FTTI (ms)']} ms (illustrative)")

# Timing check of candidate safety mechanisms: FDTI + FRTI <= FTTI
mechanisms = pd.DataFrame([
    {"SG": "SG-01", "Mechanism": "Brake-request plausibility check (independent channel)", "FDTI (ms)": 50, "FRTI (ms)": 150},
    {"SG": "SG-01", "Mechanism": "Object-list freshness / alive-counter check", "FDTI (ms)": 200, "FRTI (ms)": 150},
    {"SG": "SG-02", "Mechanism": "Camera frame-loss detection and AEB-unavailable warning", "FDTI (ms)": 100, "FRTI (ms)": 200},
])
mechanisms = mechanisms.merge(safety_goals_df[["SG", "FTTI (ms)"]], on="SG")
mechanisms["FHTI (ms)"] = mechanisms["FDTI (ms)"] + mechanisms["FRTI (ms)"]
mechanisms["Fast enough"] = mechanisms["FHTI (ms)"] <= mechanisms["FTTI (ms)"]
display(mechanisms)
for _, m in mechanisms.iterrows():
    verdict = "meets" if m["Fast enough"] else "VIOLATES"
    print(f"  {m.SG} / {m.Mechanism}: FDTI {m['FDTI (ms)']} + FRTI {m['FRTI (ms)']} = {m['FHTI (ms)']} ms "
          f"{verdict} FTTI {m['FTTI (ms)']} ms")

**Two observations.**

- The safe state of SG-01 (AEB switched off) means that AEB is unavailable, which is exactly the hazard of SG-02. For a Level 1 function this is acceptable, because the driver remains responsible and SG-02 has the lower ASIL. For a Level 3+ function, where no driver is available as a fallback, switching off is not a safe state, and the item would need fail-operational behaviour (emergency operation).
- The timing check is the first quantitative link between the HARA and the technical safety concept. A mechanism that violates the FTTI must be made faster or replaced.

## 10. ASIL decomposition: worked example (D → B(D) + B(D))

**ASIL decomposition** (ISO 26262-9:2018, clause 5) splits a safety requirement onto **redundant, sufficiently independent** elements, each of which then needs a lower ASIL. The original ASIL is written in brackets, e.g. **B(D)**, because some activities still refer to it.

Permitted decompositions of ASIL D: **C(D) + A(D)**, **B(D) + B(D)**, **D(D) + QM(D)**.

**Example.** The ASIL D requirement derived from SG-01, "*A deceleration request shall only be issued when a collision threat exists*", is decomposed as follows:

| Element | Requirement | ASIL |
|---|---|---|
| Decision channel (ADAS ECU) | Issue a deceleration request only when the fused object list indicates an imminent collision | B(D) |
| Plausibility channel (brake ECU) | Forward a deceleration request only when an independent check (e.g. raw radar range-rate and vehicle speed) confirms an imminent collision | B(D) |

Unintended braking now requires **both** channels to fail in the same way at the same time.

**Conditions an assessor will check:**

- **Independence** is demonstrated by a **dependent failure analysis** (DFA, ISO 26262-9:2018, clause 7): no common cause (shared power supply, clock, sensor, software library, compiler, development team) and no cascading failure from one channel to the other.
- **Shared sensors defeat the decomposition.** If both channels use the same radar, a corrupted radar target is a common-cause fault, and the radar inherits ASIL D (or needs its own diverse confirmation).
- **Hardware metrics and the PMHF** of the safety goal are still evaluated against the **ASIL D** targets ([Notebook 03](03_fmea_and_fmeda.ipynb)). Decomposition does not relax them.
- **Side effect on the other goal:** an AND-gate against false braking makes *omission* slightly more likely (either channel can veto). SG-02 must be re-checked.

In [ ]:
# ISO 26262-9:2018 clause 5: permitted decompositions (identical to avsafety.iso26262.decomposition_options)
DECOMPOSITIONS = {"D": [("C", "A"), ("B", "B"), ("D", "QM")], "C": [("B", "A"), ("C", "QM")],
                  "B": [("A", "A"), ("B", "QM")], "A": [("A", "QM")]}


def decomposition_options(asil: str) -> list:
    """Valid decompositions in ISO notation, e.g. ('B(D)', 'B(D)')."""
    if asil not in DECOMPOSITIONS:
        raise ValueError(f"ASIL {asil!r} cannot be decomposed")
    return [(f"{a}({asil})", f"{b}({asil})") for a, b in DECOMPOSITIONS[asil]]


sg01_asil = safety_goals_df.loc[safety_goals_df.SG == "SG-01", "ASIL"].item()
print(f"SG-01 has ASIL {sg01_asil}; permitted decompositions:")
for a, b in decomposition_options(sg01_asil):
    print(f"  {a} + {b}")
chosen = ("B(D)", "B(D)")
assert chosen in decomposition_options(sg01_asil)
print(f"Chosen: {chosen[0]} decision channel + {chosen[1]} plausibility channel")

# A minimal dependent-failure checklist for the two channels.
# Each entry: (coupling factor, shared by both channels?) -- illustrative architecture
dfa = pd.DataFrame([
    ("Power supply", False), ("Clock / timing source", False), ("Front radar sensor", True),
    ("Vehicle-speed signal", True), ("Software platform / OS", False), ("Development team and tools", False),
], columns=["Coupling factor", "Shared"])
display(dfa)
shared = dfa.loc[dfa.Shared, "Coupling factor"].tolist()
if shared:
    print(f"DFA finding: {len(shared)} shared coupling factor(s): {', '.join(shared)}.")
    print("  Each must be developed to the ASIL before decomposition (D) or its faults must be detected"
          " independently; otherwise the B(D) + B(D) argument does not hold.")
else:
    print("DFA: no shared coupling factors identified in this checklist.")

## 11. HARA worksheet

The class below is a reusable worksheet: it records rationale, rejects invalid classes and gives each safety goal the highest ASIL of the hazardous events it covers. The example is a lane keeping assist (LKA) function.

> **Illustrative data:** teaching ratings, not a real HARA.

In [ ]:
class HARAWorksheet:
    """HARA worksheet: hazardous events with rationale, and safety goals that inherit the highest ASIL."""

    def __init__(self, item_name: str):
        self.item_name = item_name
        self.events = []
        self.safety_goals = []

    def add_hazardous_event(self, he_id, malfunctioning_behaviour, situation, severity, exposure,
                            controllability, rationale=""):
        asil = determine_asil(severity, exposure, controllability)   # raises on invalid classes
        self.events.append({"HE": he_id, "Malfunctioning behaviour": malfunctioning_behaviour,
                            "Situation": situation, "S": severity, "E": exposure, "C": controllability,
                            "ASIL": asil, "Rationale": rationale})
        print(f"Added {he_id}: {malfunctioning_behaviour} -> {asil}")
        return asil

    def add_safety_goal(self, sg_id, he_ids, statement, safe_state, ftti_ms=None):
        known = {e["HE"]: e["ASIL"] for e in self.events}
        missing = [h for h in he_ids if h not in known]
        if missing:
            raise KeyError(f"unknown hazardous event(s): {missing}")
        asil = max((known[h] for h in he_ids), key=asil_rank)
        if asil == "QM":
            print(f"Note: {sg_id} covers only QM events; no ASIL safety goal is required.")
        self.safety_goals.append({"SG": sg_id, "Covers": ", ".join(he_ids), "Safety goal": statement,
                                  "ASIL": asil, "Safe state": safe_state, "FTTI (ms)": ftti_ms})

    def events_df(self):
        return pd.DataFrame(self.events)

    def goals_df(self):
        return pd.DataFrame(self.safety_goals)

    def summary(self):
        print(f"\nHARA summary for: {self.item_name}")
        print("=" * 60)
        print(f"Hazardous events: {len(self.events)}")
        counts = pd.Series([e["ASIL"] for e in self.events]).value_counts()
        for a in reversed(ASIL_ORDER):
            if a in counts:
                print(f"  {a}: {counts[a]}")
        print(f"Safety goals: {len(self.safety_goals)}")
        uncovered = [e["HE"] for e in self.events if e["ASIL"] != "QM"
                     and not any(e["HE"] in g["Covers"].split(", ") for g in self.safety_goals)]
        print(f"ASIL-rated events without a safety goal: {uncovered or 'none'}")


worksheet = HARAWorksheet("Lane keeping assist (LKA)")
worksheet.add_hazardous_event("HE-LKA-01", "Unintended LKA steering torque leading to lane departure",
                              "Motorway, 120 km/h, dense traffic", "S3", "E3", "C2",
                              "Departure into adjacent lane at speed; driver can usually override the torque")
worksheet.add_hazardous_event("HE-LKA-02", "Unintended steering torque towards oncoming traffic",
                              "Rural road, 80 km/h, oncoming traffic", "S3", "E2", "C3",
                              "Head-on collision; little time to react")
worksheet.add_hazardous_event("HE-LKA-03", "Omission of the LKA steering correction",
                              "Motorway, 120 km/h, driver inattentive", "S2", "E3", "C1",
                              "Driver remains responsible for lateral control; omission is simply controllable")
worksheet.add_safety_goal("SG-LKA-01", ["HE-LKA-01", "HE-LKA-02"],
                          "Avoid unintended LKA steering torque above the overridable limit",
                          "LKA torque request removed, driver warned", ftti_ms=200)
worksheet.summary()
display(worksheet.goals_df())

## 12. HARA best practices

### Common mistakes to avoid

1. **Rating causes instead of behaviour**: "false negative of the CNN" is a cause; "no AEB braking when required" is the malfunctioning behaviour.
2. **Crediting the item's own safety mechanisms** in S, E or C. The HARA rates the item without them.
3. **Inconsistent S/E/C criteria**: use one rating guideline (e.g. SAE J2980) across all events.
4. **Missing operational situations**, especially those involving *other* road users (following traffic, cyclists).
5. **Over- or under-conservative ratings**: rate the worst *credible* situation, with documented rationale; neither pessimism nor optimism replaces evidence.
6. **Ignoring controllability by others**: for unintended braking, the following driver controls the outcome.
7. **Safe states that are interventions** ("apply emergency braking") instead of operating modes.

### Documentation required by ISO 26262-3

- Item definition (clause 5)
- Hazardous events with S, E, C, ASIL and classification rationale
- Safety goals with ASIL, and safe states and FTTI where applicable
- Traceability between hazardous events and safety goals
- Verification review of the HARA, and the confirmation review (ISO 26262-2)

A ready-to-use document template with the same AEB example is provided in [`templates/HARA_Template.md`](../templates/HARA_Template.md).

## 13. Exercise: perform your own HARA

**Task:** perform a HARA for an **adaptive cruise control (ACC)** function (speed and distance control, deceleration authority up to 3.5 m/s², 0–180 km/h).

1. Formulate vehicle-level malfunctioning behaviours. Suggested starting points:
   - Unintended acceleration
   - Unintended deceleration
   - No (or insufficient) deceleration when the lead vehicle slows down
   - Excessive deceleration
2. Combine them with operational situations and rate S, E and C, **with rationale**.
3. Determine the ASILs, formulate safety goals with a safe state and an illustrative FTTI.
4. Optional: rate the remaining AEB behaviour **MB-03** ("deceleration stronger or longer than required") from section 5.

In [ ]:
# Exercise: your HARA for ACC (the cell runs as it is; uncomment and complete the lines)
acc_hara = HARAWorksheet("Adaptive cruise control (ACC)")

# acc_hara.add_hazardous_event("HE-ACC-01", "Unintended acceleration",
#                              "Motorway, 130 km/h, slower lead vehicle", "S?", "E?", "C?", "rationale ...")
# acc_hara.add_hazardous_event("HE-ACC-02", "Unintended deceleration", "...", "S?", "E?", "C?", "...")
# acc_hara.add_hazardous_event("HE-ACC-03", "No deceleration when the lead vehicle slows", "...", "S?", "E?", "C?", "...")
# acc_hara.add_hazardous_event("HE-ACC-04", "Excessive deceleration", "...", "S?", "E?", "C?", "...")
# acc_hara.add_safety_goal("SG-ACC-01", ["HE-ACC-01"], "Avoid unintended acceleration", "ACC off, driver warned", ftti_ms=...)

acc_hara.summary()

<details>
<summary>Example solution (one possible rating; yours may differ if your rationale differs)</summary>

| HE | Malfunctioning behaviour | Situation | S | E | C | ASIL | Rationale |
|---|---|---|---|---|---|---|---|
| HE-ACC-01 | Unintended acceleration | Motorway, 130 km/h, closing on a slower lead vehicle | S3 | E3 | C2 | B | Driver can brake or override, but the closing speed is high |
| HE-ACC-02 | Unintended deceleration (up to 3.5 m/s²) | Motorway, following vehicle at short headway | S2 | E4 | C1 | A | ACC deceleration is limited; following drivers can usually react |
| HE-ACC-03 | No deceleration when the lead vehicle slows | Motorway, lead vehicle brakes moderately | S3 | E3 | C1 | A | Driver is responsible for monitoring and braking |
| HE-ACC-04 | Excessive deceleration | Rural road, following traffic | S2 | E3 | C2 | A | Similar to HE-ACC-02, at a lower exposure |

Safety goals: *SG-ACC-01 Avoid unintended acceleration [ASIL B]*, safe state "ACC switched off, driver warned"; *SG-ACC-02 Avoid unintended or excessive deceleration [ASIL A]*.
</details>

## Key takeaways

1. **The HARA works at vehicle level.** The item is the whole function (here AEB, including the brake request); hazards are malfunctioning behaviours such as omission and unintended braking.
2. **Causes are out of scope of the HARA.** Whether a missed pedestrian stems from an E/E fault (ISO 26262) or a performance limitation (ISO 21448 / ISO/PAS 8800) does not change the hazard rating, but it decides which standard treats the cause.
3. **S, E and C follow the standard's wording.** E is rated by duration or frequency; C concerns the driver *or other persons at risk*; C0 and E0 carry no percentage and no ASIL.
4. **Safety goals are vehicle-level goals** with a concrete safe state (an operating mode, not an intervention) and an FTTI. Safety mechanisms must satisfy FDTI + FRTI ≤ FTTI.
5. **For a Level 1 AEB, unintended braking usually drives the ASIL**, because the driver remains responsible when AEB merely fails to act.
6. **ASIL decomposition** (e.g. D → B(D) + B(D)) needs sufficient independence shown by a dependent failure analysis, and it leaves the hardware-metric targets at the original ASIL.

## Self-assessment

1. What is the difference between a malfunctioning behaviour, a hazard and a hazardous event?
2. Why is "Pedestrian Detection System" a poor item for a HARA?
3. A hazardous event is rated S3, E4, C3. Which ASIL results, and why is it the highest possible?
4. Compare the ASIL of AEB omission and unintended AEB braking in this notebook. Which parameter drives the difference, and why?
5. Define the FTTI. Why is "FTTI = N/A" wrong for the unintended-braking goal?
6. An engineer rates S2, E2, C2 and obtains QM. A reviewer argues that E should be E4. Recalculate the ASIL and discuss what evidence would settle the exposure rating.
7. Under which conditions may an ASIL D requirement be decomposed into B(D) + B(D)?

<details>
<summary>Answers</summary>

1. A malfunctioning behaviour is a failure or unintended behaviour of the item at vehicle level (e.g. unintended braking). A hazard is the potential source of harm it creates (e.g. rear-end collision). A hazardous event is the combination of a hazard with an operational situation (e.g. unintended full braking at 130 km/h with a close follower).
2. It contains no actuator and therefore cannot itself produce a vehicle-level hazardous event. Its outputs (false negatives, false positives) are causes, not hazards. The item should be the AEB function including the brake request.
3. ASIL D (S + E + C = 10), the top cell of Table 4: maximum severity, highest exposure, lowest controllability.
4. Omission: S3/E2/C2 → A and S3/E3/C2 → B. Unintended braking: S3/E4/C3 → D. Exposure (an unintended activation can occur at any time, so E4) and controllability (following drivers are surprised, C3) drive the difference. For omission, the Level 1 driver remains responsible for braking.
5. The FTTI is the minimum time from the occurrence of a fault in the item to a possible hazardous event, if no safety mechanism acts. A fault that commands unintended braking leads to a hazardous event after a finite time (deceleration build-up), so the FTTI exists and bounds how fast the plausibility check must react.
6. S2/E4/C2 → B. Exposure must be backed by evidence about the operational situation (field data, naturalistic-driving studies, statistics on how often or how long the situation occurs), and the basis (duration or frequency) must be stated.
7. The decomposed requirements must be allocated to redundant elements that are sufficiently independent (shown by a dependent failure analysis: no common cause, no cascading failures). Each element is developed to B(D). Hardware metrics and the PMHF of the safety goal are still evaluated against ASIL D.
</details>

## References

- ISO 26262-1:2018, -3:2018 and -9:2018, *Road vehicles: Functional safety*. Geneva: ISO.
- ISO 21448:2022, *Road vehicles: Safety of the intended functionality*. Geneva: ISO.
- ISO/PAS 8800:2024, *Road vehicles: Safety and artificial intelligence*. Geneva: ISO. https://www.iso.org/standard/83303.html
- SAE J2980, *Considerations for ISO 26262 ASIL Hazard Classification*. SAE International.
- Patel, M., Jung, R., Khatun, M. (2025). *A Systematic Literature Review on Safety of the Intended Functionality for Automated Driving Systems*. SAE Technical Paper 2025-01-5030. https://doi.org/10.4271/2025-01-5030
- Hou, Z., Liu, D., Yang, Y., Liu, H. (2024). Assessing Unknown Hazards for SOTIF Based on Twin Scenarios Empowered Autonomous Driving. *IEEE Internet of Things Journal*, 11(20), 32631–32644. https://doi.org/10.1109/JIOT.2024.3424550

## Further practice

- The HARA document template in this repository: [`templates/HARA_Template.md`](../templates/HARA_Template.md).
- A hands-on HARA exercise notebook in the author's companion repository: [AI Perception Safety Workshop, Notebook 09: HARA exercise](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/09_hara_exercise.ipynb).

---
**Navigation:** ← [01 ISO 26262 fundamentals](01_iso26262_fundamentals.ipynb) · [Module index](README.md) · [03 FMEA and FMEDA](03_fmea_and_fmeda.ipynb) →